# Returns, Characteristics, Portfolios, Factors, and Weights  

##### Rong Wang, June 2026

In [1]:
# Packages 
import numpy as np
import pandas as pd
from pathlib import Path
import os
import gc
import time
import datetime as dt
from dateutil.relativedelta import relativedelta
from pandas.tseries.offsets import *
from joblib import Parallel, delayed
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Data directory
datapath = Path('/work/rw196/data/')

### Individual stock returns and characteristics 

Individual stock characteristics data used in [Kelly, Pruitt, and Su (2019)](https://www.sciencedirect.com/science/article/abs/pii/S0304405X19301151), [Gu, Kelly, and Xiu (2020)](https://academic.oup.com/rfs/article/33/5/2223/5758276), and [Gu, Kelly, and Xiu (2021)](https://www.sciencedirect.com/science/article/abs/pii/S0304407620301998) is available on the websites of [Prof Dacheng Xiu](https://dachxiu.chicagobooth.edu/) or [Shihao Gu](https://shihaogu.com/), with the current version ending in 2021. Instead, I opt to use a different data source: [Global Factor Data](https://jkpfactors.com/stock-char) organized by Prof. Theis Jensen, Bryan Kelly, and Lasse Heje Pedersen, which offers the most up-to-date individual stock returns and firm characteristics.

The data sample ranges from Jan 1960 to Dec 2025.   

In [2]:
%%time
# Load stock returns and characteristics data (1959-2025)
charc = pd.read_parquet(datapath / 'JKP_stock_charcs.parquet')
charc = charc.rename(columns={"eom": "date"})
charc['date'] = pd.to_datetime(charc['date'])+MonthEnd(0)
charc

CPU times: user 7.73 s, sys: 3.54 s, total: 11.3 s
Wall time: 6.05 s


,excntry,date,permno,size_grp,me,ret,ret_exc,sic,age,aliq_at,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,USA,1962-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1.0,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,USA,1962-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,25.0,<NA>,...,<NA>,<NA>,0.743086,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,USA,1962-01-31,<NA>,mega,372.722332,<NA>,<NA>,<NA>,145.0,0.972656,...,<NA>,<NA>,0.735235,0.011927,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,USA,1962-01-31,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,37.0,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,USA,1962-01-31,<NA>,small,58.4999,<NA>,<NA>,<NA>,73.0,<NA>,...,<NA>,<NA>,0.489611,0.112222,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3998293,USA,2025-08-31,93436.0,mega,1076880.65763,0.083044,0.079239,3711.0,248.0,0.788127,...,-0.066389,-1.359411,0.678298,0.050371,0.034529,0.340925,11.852687,0.000951,0.001418,0.001062
3998294,USA,2025-09-30,93436.0,mega,1478249.28,0.332015,0.328706,3711.0,249.0,0.788127,...,-0.066389,-1.359411,0.678298,0.050371,0.0326,0.357704,11.852687,0.001045,0.001188,0.001102
3998295,USA,2025-10-31,93436.0,mega,1518435.92264,0.026624,0.023356,3711.0,250.0,0.780687,...,-0.0405,-0.885243,0.67445,0.048551,0.029699,0.327992,13.878411,0.001166,0.001178,0.001097
3998296,USA,2025-11-30,93436.0,mega,1430667.55923,-0.057802,-0.060814,3711.0,251.0,0.780687,...,-0.0405,-0.885243,0.67445,0.048551,0.028618,0.343989,13.878411,0.001219,0.001153,0.001123


In [3]:
# List of characteristics
charc_list = [col for col in charc.columns if col not in 
                ['excntry','date','permno','size_grp','me','ret','ret_exc','sic']]
len(charc_list)

153

In [ ]:
# # Drop stocks in the bottom 20th percentile of NYSE market capitalization
# charc = charc[charc['size_grp'].isin(['mega', 'large', 'small'])]
# charc['size_grp'].value_counts()

#### Lag characteristics by 1 period 

Portfolio assignments and factor constructions are based on 1-month lagged characteristics. Instead of looping through every single characteristic column, I shift the return and date forward by 1 month to achieve the same goal.

In [4]:
# Sort by permno and date before computing lead return
charc = charc.sort_values(['permno', 'date'])

# Now compute the 1-month lead return
charc['ret'] = charc.groupby('permno')['ret'].shift(-1)

# Move date 1M forward such that all characteristics (including size_grp, me, and sic) are 1M lagged 
charc['date'] = charc['date'] + pd.DateOffset(months=1)

In [5]:
# Drop obs without ret/permno
charc = charc.dropna(subset=['ret','permno']).reset_index(drop=True)
charc = charc.drop(columns=['excntry','ret_exc']).reset_index(drop=True)
charc

,date,permno,size_grp,me,ret,sic,age,aliq_at,aliq_mat,ami_126d,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,1986-02-28,10000.0,micro,16.1,-0.257143,3990.0,25.0,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.005173,<NA>
1,1986-03-28,10000.0,micro,11.96,0.365385,3990.0,26.0,<NA>,<NA>,<NA>,...,<NA>,<NA>,0.365969,<NA>,<NA>,<NA>,<NA>,<NA>,0.007661,<NA>
2,1986-04-30,10000.0,micro,16.33,-0.098592,3990.0,27.0,<NA>,<NA>,<NA>,...,<NA>,<NA>,0.365969,<NA>,<NA>,<NA>,<NA>,<NA>,0.007436,<NA>
3,1986-05-30,10000.0,micro,15.172,-0.222656,3990.0,28.0,<NA>,<NA>,<NA>,...,<NA>,<NA>,0.365969,<NA>,0.000787,1.086926,<NA>,0.00735,0.007654,<NA>
4,1986-06-30,10000.0,nano,11.793859,-0.005025,3990.0,29.0,<NA>,<NA>,<NA>,...,<NA>,<NA>,0.757813,<NA>,0.000763,1.12244,15.228146,0.007469,0.00713,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742063,2025-08-31,93436.0,mega,994309.16323,0.083044,3711.0,247.0,0.788127,0.10949,0.000001,...,-0.066389,-1.359411,0.678298,0.050371,0.035037,0.335132,11.852687,0.000905,0.001002,0.001016
3742064,2025-09-30,93436.0,mega,1076880.65763,0.332015,3711.0,248.0,0.788127,0.118109,0.000001,...,-0.066389,-1.359411,0.678298,0.050371,0.034529,0.340925,11.852687,0.000951,0.001418,0.001062
3742065,2025-10-30,93436.0,mega,1478249.28,0.026624,3711.0,249.0,0.788127,0.097392,0.000001,...,-0.066389,-1.359411,0.678298,0.050371,0.0326,0.357704,11.852687,0.001045,0.001188,0.001102
3742066,2025-11-30,93436.0,mega,1518435.92264,-0.057802,3711.0,250.0,0.780687,0.103827,0.000001,...,-0.0405,-0.885243,0.67445,0.048551,0.029699,0.327992,13.878411,0.001166,0.001178,0.001097


In [6]:
# Unique dates and stocks
allDates = charc['date'].dt.year * 100 + charc['date'].dt.month
allDates = allDates.drop_duplicates().sort_values().to_numpy()
allPermnos = charc['permno'].drop_duplicates().sort_values().to_numpy().astype(int)
len(allDates), len(allPermnos)

(792, 29290)

In [7]:
# All numeric months and stocks shared across all models
np.savetxt(datapath / 'allDates.csv', allDates, fmt='%d', delimiter=',')
np.savetxt(datapath / 'allPermnos.csv', allPermnos, fmt='%d', delimiter=',')

### Univariate-sorted portfolios and weights

I following the same procedure in [Jensen, Kelly, and Pedersen (2023)](https://onlinelibrary.wiley.com/doi/full/10.1111/jofi.13249) to construct portfolios and factors. See details in their paper and data appendix.

For each characteristic, I keep the two univariate-sorted corner portfolios — top and bottom decile portfolios. [Lettau and Pelger (2020)](https://academic.oup.com/rfs/article-abstract/33/5/2274/5756219) shows that most of the relevant information is contained in the extreme first and tenth decile portfolios. In summary, I have 153*2 = 306 univariate-sorted portfolios in total.

Note that the timing of my portfolio construction is slightly different from what the literature does. While Fama-French form portfolios annually at the end of each June and JKP form portfolios on a monthly basis, I construct portfolios based on the characteristics information at the end of each December to be consistent with my rolling testing periods. Consistent with JKP, I compute capped value-weighted returns and store portfolio weights on individual stocks. [Fama and French (2008)](https://onlinelibrary.wiley.com/doi/10.1111/j.1540-6261.2008.01371.x) and [Hou, Xue, and Zhang (2020)](https://academic.oup.com/rfs/article-abstract/33/5/2019/5236964?redirectedFrom=fulltext) discuss the importance of using NYSE breakpoints to make portfolio sorts robust to outliers and skewed distributions. 

Information as of December. Since I have lagged the characteristics, I extract the observations in January.

In [8]:
jan = charc[charc['date'].dt.month==1].copy().reset_index(drop=True)
# Convert everything to standard floats/objects, replacing <NA> with np.nan, while ignoring 'date' and 'size_grp'
jan[jan.columns.difference(['date','size_grp'])] = jan[jan.columns.difference(['date','size_grp'])].astype(float)
jan

,date,permno,size_grp,me,ret,sic,age,aliq_at,aliq_mat,ami_126d,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,1987-01-31,10000.0,nano,1.981547e+00,-0.212121,3990.0,36.0,NaN,NaN,1.371852e+01,...,NaN,NaN,0.571950,NaN,0.001537,1.499872,2.617846,1.317024,1.913113,0.846164
1,1987-01-31,10001.0,nano,6.937000e+00,-0.035714,4920.0,12.0,NaN,NaN,1.629137e+01,...,NaN,NaN,0.711161,NaN,0.001059,1.270361,3.312806,2.630911,6.689265,1.953846
2,1988-01-31,10001.0,nano,5.828000e+00,0.063830,4924.0,24.0,0.533838,0.538235,2.407603e+01,...,-0.003908,-0.147436,0.729152,-0.035511,0.001309,3.310462,2.765372,3.778837,5.730555,3.990824
3,1989-01-31,10001.0,nano,6.362250e+00,0.019608,4924.0,36.0,0.556474,0.620230,2.305804e+01,...,0.010908,0.226950,0.756384,0.012697,0.000302,1.857314,2.987471,8.110663,6.008550,7.809863
4,1990-01-31,10001.0,micro,1.034775e+01,-0.018519,4924.0,48.0,0.892288,0.968528,7.080035e+00,...,-0.277350,-4.262417,0.826893,0.023270,0.001707,3.721716,2.569062,1.836945,0.007790,2.421445
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311463,2021-01-31,93436.0,mega,6.773402e+05,0.124506,3711.0,192.0,0.779729,0.243884,1.480000e-06,...,-0.089619,-9.275171,0.689222,0.000259,0.066533,0.539604,5.360047,0.000273,0.000335,0.000213
311464,2022-01-31,93436.0,mega,1.092191e+06,-0.113609,3711.0,204.0,0.969883,0.052417,1.040000e-06,...,-0.045987,-1.162769,0.699649,0.006570,0.022622,0.420708,15.080915,0.000915,0.000832,0.000780
311465,2023-01-31,93436.0,mega,3.897415e+05,0.406235,3711.0,216.0,0.843883,0.041459,1.410000e-06,...,0.006787,0.048865,0.699544,0.008947,0.028228,0.394399,15.576261,0.000595,0.000331,0.000671
311466,2024-01-31,93436.0,mega,7.914088e+05,-0.246257,3711.0,228.0,0.902084,0.146506,7.400000e-07,...,0.070327,0.522427,0.685681,0.001093,0.037105,0.162204,15.119253,0.000523,0.000546,0.000459


Helper functions.

In [9]:
# Calculates breakpoints to divide data into a specified number of equal-sized groups
def get_equal_group_breakpoints(arr, num_groups: int):
    """
    This function determines the values that split a dataset into `num_groups`
    portfolios, each containing an equal number of observations.

    Args:
        arr (np.ndarray): The input 1D array of data. NaNs are ignored.
        num_groups (int): The number of equal-sized groups to create (e.g., 3 for terciles).

    Returns:
        np.ndarray: An array of the calculated breakpoints. The number of breakpoints
                    will be `num_groups` - 1.
    """
    # Remove NaNs
    clean_arr = arr[~np.isnan(arr)]
        
    # If the array is empty after cleaning, return NaNs for the breakpoints
    if len(clean_arr) == 0:
        return np.full(num_groups - 1, np.nan)

    # Sort the data
    sorted_arr = np.sort(clean_arr)
    n = len(sorted_arr)

    # Determine the desired percentiles for the breakpoints
    #    For `num_groups = 3`, we need percentiles at [33.33, 66.67]
    #    For `num_groups = 5`, we need percentiles at [20, 40, 60, 80]
    p_desired = 100 * np.arange(1, num_groups) / num_groups

    # Create the percentile ranks corresponding to each data point
    #    It considers the k-th value to be the 100*(k-0.5)/n percentile (same as Matlab convention)
    p_rank = 100 * (np.arange(1, n + 1)-0.5) / n

    # Use linear interpolation to find the data values at the desired percentiles.
    #    This finds the breakpoints.
    breakpoints = np.interp(p_desired, p_rank, sorted_arr)

    return breakpoints

In [10]:
# Calculates the weighted average of returns and the corresponding weights
def getPtfRetsWts(df: pd.DataFrame, ret_name: str, me_name: str, method: str) -> tuple[float, pd.Series]:
    """
    This function computes portfolio returns and weights using one of three schemes:
    'ew' for equal-weighted, 'vw' for value-weighted, and 'cvw' for
    capped value-weighted.

    Args:
        df (pd.DataFrame): DataFrame containing stock data.
        ret_name (str): The name of the column containing stock returns.
        me_name (str): The name of the column containing market equity values.
        method (str): The weighting method ('vw', 'ew', or 'cvw').

    Returns:
        A tuple containing:
        - float: The calculated portfolio return for the given period.
        - pd.Series: The weights for each stock in the portfolio.

    Raises:
        ValueError: If an invalid method is specified or for 'cvw' issues.
    """
    # At least 5 stocks in each of the long and short legs
    if len(df) < 5:
        return np.nan, pd.Series(dtype=float)
        
    rt = df[ret_name]
    me = df[me_name]
    
    if method == 'ew':
        # Equal weights for all stocks
        weights = pd.Series(1 / len(df), index=df.index)
        portfolio_return = rt.mean()

    elif method == 'vw':
        # Value weights based on market equity
        total_me = me.sum()
        weights = me / total_me
        portfolio_return = (rt * weights).sum()

    elif method == 'cvw':
        if 'size_grp' not in df.columns:
            raise ValueError("Data does not have size_grp indicators required for 'cvw' method.")
        
        # Cap market equity at the 80th percentile of NYSE stocks
        nyse80 = df['size_grp'] == 'mega'
        cap = me[nyse80].min() if np.any(nyse80) else np.nan  
        capped_me = me.clip(upper=cap)
        
        # Calculate weights based on capped market equity
        total_capped_me = capped_me.sum()
        weights = capped_me / total_capped_me
        portfolio_return = (rt * weights).sum()

    else:
        raise ValueError("Method must be one of 'vw', 'ew', or 'cvw'.")
        
    return portfolio_return, weights

#### Assign portfolio buckets

In [11]:
# Portfolio sorts are based on non-micro stocks (i.e., larger than NYSE 20th percentile)
non_micro = jan[jan['size_grp'].isin(['mega', 'large','small'])]

# Initialize the portfolio assigment matrix
ptfassign = jan[['date','permno']].copy()
ptfassign['year'] = ptfassign['date'].dt.year

In [12]:
%%time
# Loop through the charcs
for ch in tqdm(charc_list, desc='Processing', colour='green'):
    
    # Sort non-micro stocks into three groups of equal numbers
    breaks = non_micro.groupby(['date'])[ch].apply(lambda x: get_equal_group_breakpoints(x.values,3)).apply(pd.Series).rename(columns={0: f'{ch}1', 1: f'{ch}2'}).reset_index()
    
    # Merge breakpoints to the original data
    breaks = pd.merge(jan, breaks, how='left', on=['date'])
    
    # Assign portfolio buckets
    conditions = [
        breaks[ch]<=breaks[f'{ch}1'],
        (breaks[ch]>breaks[f'{ch}1']) & (breaks[ch]<=breaks[f'{ch}2']),
        breaks[ch]>breaks[f'{ch}2']
    ]
    choices = [1,2,3] # list(range(1, 4))
    ptfassign[ch] = np.select(conditions, choices, default=np.nan)

Processing: 100%|██████████| 153/153 [00:26<00:00,  5.75it/s]

CPU times: user 26.5 s, sys: 45.5 ms, total: 26.5 s
Wall time: 26.6 s


In [13]:
# Merge back with monthly records
charc['year'] = charc['date'].dt.year
ptfassign = pd.merge(charc[['date','permno','me','ret','size_grp','year']], ptfassign.drop(columns='date'), how='left', on=['permno','year'])
ptfassign

,date,permno,me,ret,size_grp,year,age,aliq_at,aliq_mat,ami_126d,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,1986-02-28,10000.0,16.1,-0.257143,micro,1986,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1986-03-28,10000.0,11.96,0.365385,micro,1986,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1986-04-30,10000.0,16.33,-0.098592,micro,1986,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1986-05-30,10000.0,15.172,-0.222656,micro,1986,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1986-06-30,10000.0,11.793859,-0.005025,nano,1986,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742063,2025-08-31,93436.0,994309.16323,0.083044,mega,2025,2.0,3.0,1.0,1.0,...,3.0,3.0,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0
3742064,2025-09-30,93436.0,1076880.65763,0.332015,mega,2025,2.0,3.0,1.0,1.0,...,3.0,3.0,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0
3742065,2025-10-30,93436.0,1478249.28,0.026624,mega,2025,2.0,3.0,1.0,1.0,...,3.0,3.0,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0
3742066,2025-11-30,93436.0,1518435.92264,-0.057802,mega,2025,2.0,3.0,1.0,1.0,...,3.0,3.0,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0


#### Compute portfolio returns and weights

In [14]:
def compute_charc_port(ch):

    local_returns_list = []
    local_weights_list = []
    #ptfassign_ch = ptfassign[ptfassign[ch]!=2][['date','permno','ret','me','size_grp',ch]].copy()
    ptfassign_ch = ptfassign[['date','permno','ret','me','size_grp',ch]].copy()
    groups = ptfassign_ch.dropna(subset=['ret', 'me', ch]).groupby(['date', ch])

    for group_keys, group_df in groups:
        dt, ptfid = group_keys
        group_df = group_df.set_index('permno')
        # Compute the portfolio return and its weights
        ptfret, ptfwts = getPtfRetsWts(group_df, 'ret', 'me', 'cvw')
        local_returns_list.append({
            'date': dt,
            'portfolio': f'{ch}_{ptfid.astype(int)}',
            'wret': ptfret
        })
        
        for permno, wt in ptfwts.items():
            local_weights_list.append({
                'date': dt,
                'portfolio': f'{ch}_{ptfid.astype(int)}',
                'permno': permno,
                'wts': wt
            })
            
    return local_returns_list, local_weights_list

Parallel processing saves a lot of time, but the kernel remains busy after the procedure. This is probably becasue the child processes created by joblib did not shut down cleanly, leading to a deadlock where the main kernel is stuck waiting for child processes that will never respond. Hence, I will need to manually interrupt the kernel to proceed.

In [ ]:
# %%time
# # Parallel processing
# with Parallel(n_jobs=-1) as parallel:
#     results = parallel(delayed(compute_charc_port)(ch) for ch in charc_list)

# # Combine the results from all processes
# returns_list = []
# weights_list = []
# for r_list, w_list in results:
#     returns_list.extend(r_list)
#     weights_list.extend(w_list)

In [15]:
%%time
# Store results
returns_list = []
weights_list = []

# Loop through the charcs
for ch in tqdm(charc_list, desc='Processing', colour='green'):

    # Focus on the top and bottom decile portfolios (include all three portfolios for size to replicate MKT)
    if ch == "market_equity":
        ptfassign_ch = ptfassign[['date','permno','ret','me','size_grp',ch]].copy()
    else:
        ptfassign_ch = ptfassign[ptfassign[ch]!=2][['date','permno','ret','me','size_grp',ch]].copy()

    # Group the data (dropping missing values)
    groups = ptfassign_ch.dropna(subset=['ret', 'me', ch]).groupby(['date', ch])

    # Loop through groups to get both returns and weights
    for group_keys, group_df in groups:
        dt, ptfid = group_keys
        group_df = group_df.set_index('permno')
        
        # Compute portfolio returns and weights
        ptfret, ptfwts = getPtfRetsWts(group_df, 'ret', 'me', 'cvw')
        
        # Append the aggregated return for the group
        returns_list.append({
            'date': dt,
            'portfolio': f'{ch}_{ptfid.astype(int)}',
            'wret': ptfret
        })
        
        # Append the detailed weights Series for all stocks in the group
        for permno, wt in ptfwts.items():
            weights_list.append({
                'date': dt,
                'portfolio': f'{ch}_{ptfid.astype(int)}',
                'permno': permno,
                'wts': wt
            })

Processing: 100%|██████████| 153/153 [18:42<00:00,  7.33s/it]

CPU times: user 16min 47s, sys: 1min 53s, total: 18min 40s
Wall time: 18min 42s


Portfolio returns.

In [16]:
# Concatenate to long format
returns_df = pd.DataFrame(returns_list)

# Reshape to wide format
port_unisort = returns_df.pivot(index='date', columns='portfolio', values='wret').reset_index()
port_unisort.columns.name = None

# Numeric date
port_unisort['date'] = pd.to_datetime(port_unisort['date']).dt.year * 100 + pd.to_datetime(port_unisort['date']).dt.month
port_unisort

,date,age_1,age_3,aliq_at_1,aliq_at_3,aliq_mat_1,aliq_mat_3,ami_126d_1,ami_126d_3,at_be_1,...,turnover_var_126d_1,turnover_var_126d_3,z_score_1,z_score_3,zero_trades_126d_1,zero_trades_126d_3,zero_trades_21d_1,zero_trades_21d_3,zero_trades_252d_1,zero_trades_252d_3
0,196001,-0.047123,-0.050879,-0.069465,-0.054612,-0.070714,-0.051925,-0.060434,-0.034156,-0.063613,...,-0.060873,-0.047347,NaN,NaN,-0.061801,-0.037451,-0.057924,-0.047215,-0.063311,-0.038560
1,196002,0.023038,0.005967,-0.001635,0.021950,0.010968,0.003964,0.013234,0.006590,-0.005384,...,0.011529,0.013243,NaN,NaN,0.017823,0.008045,0.016782,0.017243,0.017174,0.007009
2,196003,-0.004830,-0.021345,-0.023618,-0.016205,-0.008300,-0.050875,-0.019871,-0.011560,-0.021468,...,-0.018210,-0.016097,NaN,NaN,-0.039552,-0.008145,-0.045821,-0.002526,-0.039827,-0.008626
3,196004,-0.004856,-0.012027,-0.025617,0.003912,-0.012883,-0.013061,-0.011335,-0.010657,-0.023827,...,-0.009956,-0.018664,NaN,NaN,-0.024508,-0.007193,-0.029614,-0.002462,-0.026718,-0.002481
4,196005,0.044412,0.024034,0.019723,0.066094,0.043085,0.023692,0.040739,0.021631,0.030940,...,0.025770,0.035211,NaN,NaN,0.062396,0.022761,0.063709,0.029298,0.054453,0.022260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,202508,0.030904,0.042725,0.022476,0.041389,0.017519,0.053525,0.026876,0.077728,0.036686,...,0.034733,0.042701,0.045660,0.025831,0.044985,0.032113,0.042297,0.030175,0.045617,0.032369
788,202509,0.025438,0.017622,0.001299,0.035845,0.006572,0.040472,0.013226,0.037697,0.036725,...,0.017802,0.025324,0.035871,0.023026,0.046711,0.009506,0.049877,0.008380,0.049436,0.008589
789,202510,0.025013,-0.008983,-0.014304,0.026471,0.002320,0.006014,-0.004672,0.015120,0.018382,...,-0.006665,0.023272,-0.004143,0.017322,0.037814,-0.019154,0.036540,-0.019692,0.038529,-0.020889
790,202511,-0.024520,0.030217,0.019692,-0.006730,-0.006225,0.033495,0.013571,0.000845,0.008662,...,0.025469,-0.014345,0.026313,-0.005490,-0.009595,0.024278,-0.016091,0.022173,-0.008305,0.024939


Portfolio weights.

In [17]:
%%time
# Concatenate to long format
weights_df = pd.DataFrame(weights_list)  

# Reshape to wide format
port_unisort_wts = weights_df.pivot(index=['date','permno'], columns='portfolio', values='wts').reset_index()
port_unisort_wts.columns.name = None

# Numeric date
port_unisort_wts['date'] = pd.to_datetime(port_unisort_wts['date']).dt.year * 100 + pd.to_datetime(port_unisort_wts['date']).dt.month
port_unisort_wts

CPU times: user 5min 25s, sys: 41.1 s, total: 6min 6s
Wall time: 6min 6s


,date,permno,age_1,age_3,aliq_at_1,aliq_at_3,aliq_mat_1,aliq_mat_3,ami_126d_1,ami_126d_3,...,turnover_var_126d_1,turnover_var_126d_3,z_score_1,z_score_3,zero_trades_126d_1,zero_trades_126d_3,zero_trades_21d_1,zero_trades_21d_3,zero_trades_252d_1,zero_trades_252d_3
0,196001,10006.0,NaN,0.001461,NaN,NaN,NaN,0.006098,NaN,NaN,...,NaN,NaN,NaN,NaN,0.002329,NaN,0.002572,NaN,0.002254,NaN
1,196001,10014.0,NaN,0.000209,NaN,NaN,NaN,NaN,NaN,0.000518,...,NaN,NaN,NaN,NaN,0.000333,NaN,0.000367,NaN,0.000322,NaN
2,196001,10022.0,NaN,0.000449,NaN,NaN,NaN,NaN,NaN,0.001115,...,NaN,NaN,NaN,NaN,0.000716,NaN,0.000790,NaN,0.000693,NaN
3,196001,10030.0,NaN,0.001725,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.002750,NaN,NaN,NaN,0.002661,NaN
4,196001,10057.0,NaN,0.000575,NaN,NaN,NaN,NaN,NaN,0.001428,...,NaN,NaN,NaN,NaN,NaN,0.001033,NaN,0.000634,NaN,0.001006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3599247,202512,93374.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000775,NaN,NaN,NaN,NaN,NaN,NaN,0.000929,NaN,NaN
3599248,202512,93397.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000117,...,NaN,0.000056,NaN,NaN,NaN,0.000035,NaN,0.000035,NaN,0.000036
3599249,202512,93426.0,NaN,NaN,NaN,NaN,NaN,0.000086,NaN,0.000195,...,NaN,0.000093,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000060
3599250,202512,93427.0,NaN,NaN,NaN,0.003025,NaN,NaN,NaN,NaN,...,NaN,0.003655,NaN,0.002587,0.003237,NaN,0.003344,NaN,0.003226,NaN


In [19]:
%%time
# Verification
# Group by date and calculate the sum for each portfolio column
wts_sums = port_unisort_wts.drop(columns='permno').groupby('date').sum()

# Use np.allclose to check if all portfolio weights sum to close to 1
# This is more robust than checking for exact equality with 1
is_verified = np.allclose(wts_sums.replace(0, 1), 1)

if is_verified:
    print("✅ Success! For each date, all portfolio weights sum to 1.")
else:
    print("❌ Failure!")

✅ Success! For each date, all portfolio weights sum to 1.
CPU times: user 5.15 s, sys: 759 ms, total: 5.91 s
Wall time: 5.93 s


### Pre-specified factors and weights

#### Compute factor returns and weights

In [20]:
%%time
# Initialize the DataFrames
fac = port_unisort[['date']]
fac_wts = port_unisort_wts[['date','permno']]

# Loop through the charcs
for ch in tqdm(charc_list, desc='Processing', colour='green'):

    # Determine the long and short legs
    if port_unisort[f'{ch}_1'].mean() >= port_unisort[f'{ch}_3'].mean():
        fac[ch] = port_unisort[f'{ch}_1'] - port_unisort[f'{ch}_3']
        fac_wts[ch] = port_unisort_wts[f'{ch}_1'].fillna(0) - port_unisort_wts[f'{ch}_3'].fillna(0)
    else:
        fac[ch] = port_unisort[f'{ch}_3'] - port_unisort[f'{ch}_1']
        fac_wts[ch] = port_unisort_wts[f'{ch}_3'].fillna(0) - port_unisort_wts[f'{ch}_1'].fillna(0)
        
    # Replace 0 wts to NaN
    fac_wts.loc[fac_wts[ch] == 0, ch] = np.nan

Processing: 100%|██████████| 153/153 [00:04<00:00, 33.43it/s]

CPU times: user 10 s, sys: 329 ms, total: 10.4 s
Wall time: 4.59 s


Add a market factor.

In [21]:
mktassign = charc[['date','permno','me','ret']]
mktassign['date'] = pd.to_datetime(mktassign['date']).dt.year * 100 + pd.to_datetime(mktassign['date']).dt.month

# Value-weighted returns
mktassign['vwts'] = mktassign.groupby('date')['me'].transform(lambda x: x / x.sum())

# Market factor
mktassign['vwret'] = mktassign['ret']*mktassign['vwts']
mkt = mktassign.groupby('date')['vwret'].sum().reset_index().rename(columns={'vwret': 'mkt'})

In [22]:
# Merge into factor returns and weights
fac = pd.merge(mkt[mkt['date']>=196001], fac, on='date',how='left')
fac_wts = pd.merge(mktassign[['date','permno','vwts']].rename(columns={'vwts': 'mkt'}).dropna(subset='mkt'), fac_wts, on=['date','permno'], how='left')

#### Compare JKP factors and self-constructed factors

In [24]:
# 6 factors from JKP
jkp = pd.read_csv('../Data/JKP_factors_monthly.csv') 
jkp = jkp.pivot(index='date', columns='name', values='ret').reset_index() # long to wide
jkp = jkp[['date','market_equity','be_me','ope_be','at_gr1','ret_12_1']]
jkp['date'] = pd.to_datetime(jkp['date']).dt.year * 100 + pd.to_datetime(jkp['date']).dt.month
jkp

name,date,market_equity,be_me,ope_be,at_gr1,ret_12_1
0,192601,0.026212,NaN,NaN,NaN,NaN
1,192602,-0.026540,NaN,NaN,NaN,NaN
2,192603,-0.049909,NaN,NaN,NaN,NaN
3,192604,0.002679,NaN,NaN,NaN,NaN
4,192605,-0.006962,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
1195,202508,0.061672,0.050510,-0.013023,0.017396,-0.027374
1196,202509,0.013088,-0.012686,-0.038702,-0.016748,0.051697
1197,202510,0.005952,-0.026229,-0.050108,-0.002830,0.013774
1198,202511,0.006685,0.059289,0.036590,0.033331,-0.020069


In [25]:
# Compare to JKP factors
for ch in ['market_equity','be_me','ope_be','at_gr1','ret_12_1']:
    print(jkp.set_index('date')[ch].corr(fac.set_index('date')[ch]))

0.9611525888302977
0.9494019484384872
0.9577376628780443
0.8891195504014376
0.7877943371821329


The least correlation occurs for the momentum factor becasue JKP update MOM on a monthly basis, while I only form portfolios and factors at the end of each December.

### Characteristics Processing

#### Keep charcs whose factors are available during 1960-2025

I end up using the remaining 136 characteristics.

In [26]:
fac = fac.dropna(axis=1)
charc_list = [col for col in fac.columns if col not in ['date','mkt']]
len(charc_list)

136

In [27]:
# Update univariate-sorted portfolios returns
# cols_to_keep = [f"{ch}_1" for ch in charc_list] + [f"{ch}_3" for ch in charc_list]
cols_to_keep = (
    ["market_equity_1", "market_equity_2", "market_equity_3"] + 
    [f"{ch}_1" for ch in charc_list if ch != "market_equity"] +
    [f"{ch}_3" for ch in charc_list if ch != "market_equity"]     
)
port_unisort = port_unisort[['date'] + cols_to_keep]
port_unisort.to_csv(datapath / 'port.csv', index=False)
port_unisort

,date,market_equity_1,market_equity_2,market_equity_3,age_1,aliq_at_1,aliq_mat_1,ami_126d_1,at_be_1,at_gr1_1,...,seas_6_10na_3,taccruals_at_3,taccruals_ni_3,tangibility_3,tax_gr1a_3,turnover_126d_3,turnover_var_126d_3,zero_trades_126d_3,zero_trades_21d_3,zero_trades_252d_3
0,196001,-0.032444,-0.052783,-0.056936,-0.047123,-0.069465,-0.070714,-0.060434,-0.063613,-0.058126,...,-0.072832,-0.054995,-0.059599,-0.072896,-0.050391,-0.058469,-0.047347,-0.037451,-0.047215,-0.038560
1,196002,0.012878,0.010427,0.012455,0.023038,-0.001635,0.010968,0.013234,-0.005384,-0.000791,...,0.003133,-0.000677,-0.001006,-0.004301,0.030991,0.017254,0.013243,0.008045,0.017243,0.007009
2,196003,-0.029677,-0.020715,-0.014067,-0.004830,-0.023618,-0.008300,-0.019871,-0.021468,-0.027287,...,-0.029710,-0.014977,-0.020626,-0.022556,-0.005347,-0.041098,-0.016097,-0.008145,-0.002526,-0.008626
3,196004,-0.023963,-0.012592,-0.009588,-0.004856,-0.025617,-0.012883,-0.011335,-0.023827,-0.012582,...,-0.024868,-0.019934,-0.014938,-0.024167,0.001434,-0.023648,-0.018664,-0.007193,-0.002462,-0.002481
4,196005,0.028790,0.031080,0.034832,0.044412,0.019723,0.043085,0.040739,0.030940,0.020259,...,0.032795,0.037129,0.036416,0.022076,0.056598,0.059576,0.035211,0.022761,0.029298,0.022260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,202508,0.087716,0.054055,0.027688,0.030904,0.022476,0.017519,0.026876,0.036686,0.047682,...,0.041644,0.041345,0.036752,0.049784,0.032073,0.044975,0.042701,0.032113,0.030175,0.032369
788,202509,0.050443,0.019288,0.015177,0.025438,0.001299,0.006572,0.013226,0.036725,0.026390,...,0.023442,0.020081,0.020021,0.043533,0.026567,0.046697,0.025324,0.009506,0.008380,0.008589
789,202510,0.024285,0.002634,-0.002884,0.025013,-0.014304,0.002320,-0.004672,0.018382,0.004299,...,0.000435,0.000443,0.003041,0.018696,-0.002299,0.037804,0.023272,-0.019154,-0.019692,-0.020889
790,202511,0.005972,0.010773,0.014279,-0.024520,0.019692,-0.006225,0.013571,0.008662,0.015241,...,0.002553,0.008703,0.005530,0.009609,0.012810,-0.009612,-0.014345,0.024278,0.022173,0.024939


In [28]:
# Update univariate-sorted portfolios weights
port_unisort_wts = port_unisort_wts[['date','permno'] + cols_to_keep]
port_unisort_wts.to_parquet(datapath / 'port_wts.parquet', engine='pyarrow')
port_unisort_wts

,date,permno,market_equity_1,market_equity_2,market_equity_3,age_1,aliq_at_1,aliq_mat_1,ami_126d_1,at_be_1,...,seas_6_10na_3,taccruals_at_3,taccruals_ni_3,tangibility_3,tax_gr1a_3,turnover_126d_3,turnover_var_126d_3,zero_trades_126d_3,zero_trades_21d_3,zero_trades_252d_3
0,196001,10006.0,NaN,0.002285,NaN,NaN,NaN,NaN,NaN,0.003561,...,0.001856,NaN,0.003679,0.003253,NaN,0.002582,NaN,NaN,NaN,NaN
1,196001,10014.0,0.000772,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000369,NaN,NaN,NaN,NaN
2,196001,10022.0,0.001661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000793,NaN,NaN,NaN,NaN
3,196001,10030.0,NaN,0.002698,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.003049,NaN,NaN,NaN,NaN
4,196001,10057.0,0.002127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000730,NaN,NaN,NaN,NaN,NaN,NaN,0.001033,0.000634,0.001006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3599247,202512,93374.0,NaN,0.001789,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000929,NaN
3599248,202512,93397.0,0.000134,NaN,NaN,NaN,NaN,NaN,NaN,0.000046,...,NaN,0.000042,0.000042,NaN,NaN,NaN,0.000056,0.000035,0.000035,0.000036
3599249,202512,93426.0,0.000223,NaN,NaN,NaN,NaN,NaN,NaN,0.000077,...,0.000093,0.000070,0.000069,0.000085,0.000065,NaN,0.000093,NaN,NaN,0.000060
3599250,202512,93427.0,NaN,0.004394,NaN,NaN,NaN,NaN,NaN,0.003043,...,0.003675,0.002753,0.002731,0.003332,NaN,0.003236,0.003655,NaN,NaN,NaN


In [29]:
# Update factor returns
fac.to_csv(datapath / 'fac.csv', index=False)
fac

,date,mkt,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,...,seas_6_10na,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,196001,-0.066345,-0.003756,-0.014853,0.018789,0.026279,0.000489,-0.002315,0.029815,-0.014679,...,0.036183,-0.016547,-0.011525,-0.011999,0.019619,0.013154,0.013526,0.024350,0.010709,0.024750
1,196002,0.014458,-0.017072,-0.023585,-0.007004,-0.006644,0.027606,-0.020740,-0.013401,0.023048,...,0.009500,0.014435,0.019700,-0.022139,0.033298,-0.003324,0.001714,-0.009778,0.000460,-0.010165
2,196003,-0.012701,-0.016515,-0.007414,-0.042575,0.008311,-0.001860,-0.020210,-0.009227,-0.026462,...,0.016701,-0.028998,-0.016606,-0.005740,0.028206,0.040247,0.002112,0.031407,0.043295,0.031202
3,196004,-0.015198,-0.007171,-0.029529,-0.000178,0.000678,0.015998,-0.010833,0.000734,0.000836,...,0.017389,0.006836,0.005984,-0.017883,0.029920,0.021137,-0.008707,0.017315,0.027152,0.024237
4,196005,0.034136,-0.020378,-0.046372,-0.019393,-0.019109,0.008873,-0.023108,-0.050085,0.041282,...,-0.006941,-0.011537,-0.006876,-0.032027,0.030654,-0.035536,0.009441,-0.039635,-0.034411,-0.032194
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,202508,0.024079,0.011821,-0.018912,0.036006,0.050852,0.006704,0.012011,0.037876,-0.002666,...,0.002930,0.001350,0.006109,0.027553,-0.003395,-0.012855,0.007968,-0.012872,-0.012122,-0.013248
788,202509,0.03733,-0.007817,-0.034546,0.033900,0.024470,-0.029820,0.003828,-0.010474,-0.017741,...,-0.004509,0.006071,-0.001532,0.044169,0.013772,-0.037186,0.007522,-0.037205,-0.041497,-0.040847
789,202510,0.020694,-0.033996,-0.040775,0.003694,0.019791,-0.025419,-0.002676,-0.029733,0.001332,...,-0.003284,0.006215,-0.002080,0.035479,-0.006640,-0.056958,0.029937,-0.056969,-0.056231,-0.059418
790,202511,0.001428,0.054737,0.026422,0.039720,-0.012726,0.003141,0.019688,0.045449,-0.018064,...,0.027494,0.002085,0.012829,-0.001440,-0.008525,0.033908,-0.039814,0.033873,0.038264,0.033244


In [30]:
# Update factor weights
fac_wts = fac_wts[['date','permno','mkt'] + charc_list]
fac_wts.to_parquet(datapath / 'fac_wts.parquet', engine='pyarrow')
fac_wts

,date,permno,mkt,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,...,seas_6_10na,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,198602,10000.0,0.000007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,198603,10000.0,0.000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,198604,10000.0,0.000007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,198605,10000.0,0.000006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,198606,10000.0,0.000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3741039,202508,93436.0,0.014774,NaN,-0.004119,-0.003049,-0.001712,-0.004218,-0.003565,-0.002984,...,-0.004880,-0.003646,-0.003618,0.004594,-0.003849,-0.004533,-0.002490,-0.004533,-0.004607,-0.004528
3741040,202509,93436.0,0.015644,NaN,-0.004040,-0.003031,-0.001707,-0.004083,-0.003476,-0.002965,...,-0.004775,-0.003579,-0.003560,0.004511,-0.003784,-0.004334,-0.002450,-0.004335,-0.004479,-0.004336
3741041,202510,93436.0,0.0207,NaN,-0.003936,-0.003032,-0.001704,-0.003925,-0.003428,-0.002915,...,-0.004690,-0.003531,-0.003506,0.004330,-0.003720,-0.004259,-0.002424,-0.004259,-0.004392,-0.004231
3741042,202511,93436.0,0.02085,NaN,-0.003906,-0.003015,-0.001705,-0.003912,-0.003374,-0.002865,...,-0.004672,-0.003488,-0.003458,0.004291,-0.003708,-0.004093,-0.002426,-0.004094,-0.004217,-0.004088


In [31]:
# Update individual characteristics
charc = charc[['date','permno','size_grp','me','sic','ret']+charc_list]
charc['date'] = charc['date'].dt.year * 100 + charc['date'].dt.month

In [32]:
# Market capitalization
me = charc[['date','permno','me']].copy()
me = me.pivot(index='date', columns='permno', values='me') 
me.to_parquet(datapath / 'me.parquet', engine='pyarrow')

In [33]:
# Size groups
size_grp = charc[['date','permno','size_grp']].copy()
size_grp.to_parquet(datapath / 'size_grp.parquet', engine='pyarrow')

In [34]:
# Individual returns
ret_long = charc[['date','permno','ret']].copy()
ret_long.to_parquet(datapath / 'ret_long.parquet', engine='pyarrow')
ret = ret_long.pivot(index='date', columns='permno', values='ret') 
ret.to_parquet(datapath / 'ret.parquet', engine='pyarrow')
ret

permno,10000.0,10001.0,10002.0,10003.0,10005.0,10006.0,10007.0,10008.0,10009.0,10010.0,...,93426.0,93427.0,93428.0,93429.0,93430.0,93432.0,93433.0,93434.0,93435.0,93436.0
date,,,,,,,,,,,,,,,,,,,,,
196001,<NA>,<NA>,<NA>,<NA>,<NA>,0.005155,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
196002,<NA>,<NA>,<NA>,<NA>,<NA>,0.046186,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
196003,<NA>,<NA>,<NA>,<NA>,<NA>,-0.059553,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
196004,<NA>,<NA>,<NA>,<NA>,<NA>,-0.081794,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
196005,<NA>,<NA>,<NA>,<NA>,<NA>,0.048732,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202508,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.071644,0.023353,<NA>,<NA>,<NA>,<NA>,<NA>,-0.897225,<NA>,0.083044
202509,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.127727,0.100607,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.332015
202510,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,0.168799,0.208299,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.026624


In [35]:
# Save the raw characteristics without transformating or filling NaNs
charc = charc[['date','permno','sic']+charc_list]
charc.to_parquet(datapath / 'charc_raw.parquet', engine='pyarrow')
charc

,date,permno,sic,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,...,seas_6_10na,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,198602,10000.0,3990.0,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.005173,<NA>
1,198603,10000.0,3990.0,26.0,<NA>,<NA>,<NA>,<NA>,<NA>,0.060201,...,<NA>,<NA>,<NA>,0.365969,<NA>,<NA>,<NA>,<NA>,0.007661,<NA>
2,198604,10000.0,3990.0,27.0,<NA>,<NA>,<NA>,<NA>,<NA>,0.044091,...,<NA>,<NA>,<NA>,0.365969,<NA>,<NA>,<NA>,<NA>,0.007436,<NA>
3,198605,10000.0,3990.0,28.0,<NA>,<NA>,<NA>,<NA>,<NA>,0.047456,...,<NA>,<NA>,<NA>,0.365969,<NA>,0.000787,1.086926,0.00735,0.007654,<NA>
4,198606,10000.0,3990.0,29.0,<NA>,<NA>,<NA>,1.835994,<NA>,0.107259,...,<NA>,<NA>,<NA>,0.757813,<NA>,0.000763,1.12244,0.007469,0.00713,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742063,202508,93436.0,3711.0,247.0,0.788127,0.10949,0.000001,1.6759,0.145432,0.125827,...,0.045764,-0.066389,-1.359411,0.678298,0.050371,0.035037,0.335132,0.000905,0.001002,0.001016
3742064,202509,93436.0,3711.0,248.0,0.788127,0.118109,0.000001,1.6759,0.145432,0.116179,...,0.060797,-0.066389,-1.359411,0.678298,0.050371,0.034529,0.340925,0.000951,0.001418,0.001062
3742065,202510,93436.0,3711.0,249.0,0.788127,0.097392,0.000001,1.6759,0.145432,0.084635,...,0.049369,-0.066389,-1.359411,0.678298,0.050371,0.0326,0.357704,0.001045,0.001188,0.001102
3742066,202511,93436.0,3711.0,250.0,0.780687,0.103827,0.000001,1.66292,0.139455,0.084671,...,0.055516,-0.0405,-0.885243,0.67445,0.048551,0.029699,0.327992,0.001166,0.001178,0.001097


#### Fill missing characteristics

In [36]:
#charc = pd.read_parquet(datapath / 'charc_raw.parquet')
charc_list = [col for col in charc.columns if col not in ['date','permno','sic']]
charc

,date,permno,sic,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,...,seas_6_10na,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,198602,10000.0,3990.0,25.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.005173,<NA>
1,198603,10000.0,3990.0,26.0,<NA>,<NA>,<NA>,<NA>,<NA>,0.060201,...,<NA>,<NA>,<NA>,0.365969,<NA>,<NA>,<NA>,<NA>,0.007661,<NA>
2,198604,10000.0,3990.0,27.0,<NA>,<NA>,<NA>,<NA>,<NA>,0.044091,...,<NA>,<NA>,<NA>,0.365969,<NA>,<NA>,<NA>,<NA>,0.007436,<NA>
3,198605,10000.0,3990.0,28.0,<NA>,<NA>,<NA>,<NA>,<NA>,0.047456,...,<NA>,<NA>,<NA>,0.365969,<NA>,0.000787,1.086926,0.00735,0.007654,<NA>
4,198606,10000.0,3990.0,29.0,<NA>,<NA>,<NA>,1.835994,<NA>,0.107259,...,<NA>,<NA>,<NA>,0.757813,<NA>,0.000763,1.12244,0.007469,0.00713,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742063,202508,93436.0,3711.0,247.0,0.788127,0.10949,0.000001,1.6759,0.145432,0.125827,...,0.045764,-0.066389,-1.359411,0.678298,0.050371,0.035037,0.335132,0.000905,0.001002,0.001016
3742064,202509,93436.0,3711.0,248.0,0.788127,0.118109,0.000001,1.6759,0.145432,0.116179,...,0.060797,-0.066389,-1.359411,0.678298,0.050371,0.034529,0.340925,0.000951,0.001418,0.001062
3742065,202510,93436.0,3711.0,249.0,0.788127,0.097392,0.000001,1.6759,0.145432,0.084635,...,0.049369,-0.066389,-1.359411,0.678298,0.050371,0.0326,0.357704,0.001045,0.001188,0.001102
3742066,202511,93436.0,3711.0,250.0,0.780687,0.103827,0.000001,1.66292,0.139455,0.084671,...,0.055516,-0.0405,-0.885243,0.67445,0.048551,0.029699,0.327992,0.001166,0.001178,0.001097


In [ ]:
# Inspect characteristics with the most missing values
# charc[charc_list].isna().sum().sort_values(ascending = False).head(10)

Fill missing values with the _cross-sectional median_ by industry at each month. 

In [37]:
%%time
# Create 'sic2' and fill missing values with 99 (nonclassifiable establishmnt)
charc['sic2'] = pd.to_numeric(charc['sic'], errors='coerce').floordiv(100).astype('Int64')
charc['sic2'] = charc['sic2'].fillna(99) 

def fill_missing_charc(df, char_name):
    # Create a copy to avoid modifying the original slice
    char_series = df[char_name].copy()
    # Step 1: Fill NaNs using (date, sic2) medians
    fill_values = df.loc[df['sic2'] != 99].groupby(['date', 'sic2'])[char_name].transform('median')
    char_series.fillna(fill_values, inplace=True)
    # Step 2: Fill any remaining NaNs using the cross-sectional median
    fill_values_cs = df.groupby('date')[char_name].transform('median')
    char_series.fillna(fill_values_cs, inplace=True)
    return char_series

# Parallel processing
with Parallel(n_jobs=-1) as parallel:
    results_list = parallel(delayed(fill_missing_charc)(charc, ch) for ch in charc_list)
                            
charc_filled = pd.concat(results_list, axis=1)
charc[charc_list] = charc_filled

CPU times: user 5.18 s, sys: 4.86 s, total: 10 s
Wall time: 1min 16s


In [ ]:
# %%time
# # Fill NaNs simply with cross-sectional median
# for ch in tqdm(charc_list, desc='Processing', colour='green'):
#      charc[ch] = charc.groupby('date')[ch].transform(lambda x: x.fillna(x.median()))

In [38]:
# NO missing characteristics after the nan-filling
assert charc[charc_list].isna().sum().sum() == 0, "There are still missing values in some characteristics!"

#### Transform the stock characteristics

[Kelly, Pruitt, and Su (2019)](https://www.sciencedirect.com/science/article/abs/pii/S0304405X19301151), [Gu, Kelly, and Xiu (2021)](https://www.sciencedirect.com/science/article/abs/pii/S0304407620301998), and some other papers map the characteristics into the $(-1,1)$ intervel. [Kozak, Nagel, and Santosh (2020)](https://www.sciencedirect.com/science/article/abs/pii/S0304405X19301655) further normalize the ranks such that the characteristics-managed portfolios are zero-investment and have fixed leverage. 

1. For each period, rank all stock characteristics across the cross-section with ties. This transformation helps reduce the influence of outliers on the results.
2. Normalize each rank-transformed characteristic by first subtracting its cross-sectional mean and then dividing by the sum of the absolute deviations from this mean across all stocks. This ensures the resulting portfolio is zero-investment and fully allocated, with weights summing in absolute value to one (i.e., 50% long and 50% short, 1x gross leverage).

Since I follow the first approach that simply transforms the characteristics into the $(-1,1)$ intervel for two reasons:
- Since the characteristics are mainly used for estimation, it is not necessary to produce interpretable characteristics-managed portfolios. These portfolios may not be economically interesting because they consist of all existing stocks with minimal weights.
- KNS procedure produces very small values after transformation as they can be interpreted as portfolio weights. The tiny characteristics values unstablize model estimation. 

Note that the transformed characteristics are only used for model estimation. 

In [39]:
%%time
def rank_and_normalize_by_group(df, char_name):
    # Group by date FIRST, then rank the specific characteristic series
    ranks = df.groupby('date')[char_name].rank(pct=True)
    # Normalize the ranked series
    normalized_series = 2 * (ranks - 0.5)
    return normalized_series

# Process in parallel
with Parallel(n_jobs=-1) as parallel:
    results_list = parallel(delayed(rank_and_normalize_by_group)(charc, ch) for ch in charc_list)
                            
charc_filled = pd.concat(results_list, axis=1)
charc[charc_list] = charc_filled
charc

CPU times: user 4.48 s, sys: 10.5 s, total: 15 s
Wall time: 21.9 s


,date,permno,sic,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_21d,zero_trades_252d,sic2
0,198602,10000.0,3990.0,-0.824155,0.352196,0.540709,0.209628,0.136993,-0.120101,-0.143074,...,-0.250169,-0.188682,0.330912,-0.554899,-0.095439,0.024493,0.108953,-0.139189,0.245777,39
1,198603,10000.0,3990.0,-0.815089,0.437016,0.559279,0.110643,0.162176,0.181711,-0.98518,...,-0.221455,-0.181711,-0.837656,-0.483833,-0.053048,0.062142,0.039576,0.206804,0.082014,39
2,198604,10000.0,3990.0,-0.805042,0.446555,0.457647,0.155126,0.164874,0.223697,-0.991597,...,-0.264034,-0.286218,-0.837647,-0.292269,0.110084,0.071765,0.101345,0.216471,0.100336,39
3,198605,10000.0,3990.0,-0.790163,0.237032,0.445526,0.016115,-0.035253,-0.275642,-0.986906,...,-0.302166,-0.266241,-0.891892,0.431089,-0.433943,-0.362766,-0.091825,0.216048,0.285714,39
4,198606,10000.0,3990.0,-0.772061,0.300866,0.523643,0.225941,-0.251415,-0.288878,-0.946054,...,-0.38345,-0.394439,0.559774,0.433733,-0.465201,-0.315684,-0.078588,0.120213,0.142025,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742063,202508,93436.0,3711.0,0.096862,0.602092,-0.880335,-0.999163,-0.370293,0.545607,-0.914644,...,-0.041841,-0.423013,0.342259,0.949372,0.712552,-0.991213,-0.712971,-0.693305,-0.694979,37
3742064,202509,93436.0,3711.0,0.103801,0.599415,-0.859649,-0.999165,-0.371345,0.538847,-0.914369,...,-0.042607,-0.427736,0.345865,0.949039,0.703843,-0.987886,-0.703843,-0.626149,-0.686717,37
3742065,202510,93436.0,3711.0,0.109144,0.597584,-0.900854,-0.999167,-0.37388,0.535097,-0.931681,...,-0.042699,-0.429286,0.349719,0.948344,0.663403,-0.976255,-0.663403,-0.63216,-0.672152,37
3742066,202511,93436.0,3711.0,0.110208,0.560417,-0.895417,-0.999167,-0.379583,0.494167,-0.935833,...,0.16,-0.185417,0.347917,0.951667,0.61875,-0.987917,-0.61875,-0.617083,-0.655417,37


In [ ]:
# %%time
# # KNS procedure
# def scale_and_normalize(s, ch):
#     """
#     Performs a two-step normalization on a pandas Series. 
#     -> s_normalized = (rk - mean(rk)) / sum(abs(rk - mean(rk)))
#     1. Rank the characteristic with ties.
#     2. Centers the ranked characteristic and divides by the sum of absolute deviations.
#     """
#     # Step 1: cross-sectionally rank the characteristic with ties
#     charc_ranks = s.rank(method='average')

#     # Step 2: Normalize by sum of absolute deviations
#     mean_ranks = charc_ranks.mean()
#     centered_ranks = charc_ranks - mean_ranks
#     scaler = np.abs(centered_ranks).sum()
    
#     # Handle the edge case where all values are the same (scaler is zero)
#     if scaler == 0:
#         print(f"Warning: Characteristic '{ch}' has all identical values for this date group. Returning NaNs.")
#         return np.full(s.shape, np.nan)
#     else:
#         return centered_ranks / scaler

# # Apply scaling and normalization
# for ch in tqdm(charc_list, desc='Processing', colour='green'):
#     charc[ch] = charc.groupby('date')[ch].transform(lambda x: scale_and_normalize(x, ch))

In [ ]:
# # Verification
# # Group by date and calculate the sum for each characteristic column
# wts_sums = charc.groupby('date')[charc_list].sum()
# wts_abs_sums = charc.groupby('date')[charc_list].apply(lambda x: x.abs().sum())

# # Use np.allclose to check if all values in the DataFrame are close to zero.
# # This is more robust than checking for exact equality with 0
# is_verified = np.allclose(wts_sums, 0) and np.allclose(wts_abs_sums, 1)

# if is_verified:
#     print("✅ Success! For each date, all columns sum to 0 AND their absolute values sum to 1.")
# else:
#     print("❌ Failure! One or both conditions were not met.")

In [45]:
# Save the characteristics after transformation
charc = charc.drop(columns=['sic','sic2'])
charc.to_parquet(datapath / 'charc.parquet', engine='pyarrow')
charc

,date,permno,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,...,seas_6_10na,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,198602,10000.0,-0.824155,0.352196,0.540709,0.209628,0.136993,-0.120101,-0.143074,0.466385,...,-0.338007,-0.250169,-0.188682,0.330912,-0.554899,-0.095439,0.024493,0.108953,-0.139189,0.245777
1,198603,10000.0,-0.815089,0.437016,0.559279,0.110643,0.162176,0.181711,-0.98518,0.471708,...,0.010273,-0.221455,-0.181711,-0.837656,-0.483833,-0.053048,0.062142,0.039576,0.206804,0.082014
2,198604,10000.0,-0.805042,0.446555,0.457647,0.155126,0.164874,0.223697,-0.991597,0.455966,...,0.082521,-0.264034,-0.286218,-0.837647,-0.292269,0.110084,0.071765,0.101345,0.216471,0.100336
3,198605,10000.0,-0.790163,0.237032,0.445526,0.016115,-0.035253,-0.275642,-0.986906,0.373678,...,-0.030217,-0.302166,-0.266241,-0.891892,0.431089,-0.433943,-0.362766,-0.091825,0.216048,0.285714
4,198606,10000.0,-0.772061,0.300866,0.523643,0.225941,-0.251415,-0.288878,-0.946054,-0.62704,...,0.21362,-0.38345,-0.394439,0.559774,0.433733,-0.465201,-0.315684,-0.078588,0.120213,0.142025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742063,202508,93436.0,0.096862,0.602092,-0.880335,-0.999163,-0.370293,0.545607,-0.914644,0.428452,...,0.969038,-0.041841,-0.423013,0.342259,0.949372,0.712552,-0.991213,-0.712971,-0.693305,-0.694979
3742064,202509,93436.0,0.103801,0.599415,-0.859649,-0.999165,-0.371345,0.538847,-0.914369,0.428989,...,0.983292,-0.042607,-0.427736,0.345865,0.949039,0.703843,-0.987886,-0.703843,-0.626149,-0.686717
3742065,202510,93436.0,0.109144,0.597584,-0.900854,-0.999167,-0.37388,0.535097,-0.931681,0.430119,...,0.959175,-0.042699,-0.429286,0.349719,0.948344,0.663403,-0.976255,-0.663403,-0.63216,-0.672152
3742066,202511,93436.0,0.110208,0.560417,-0.895417,-0.999167,-0.379583,0.494167,-0.935833,0.379583,...,0.97875,0.16,-0.185417,0.347917,0.951667,0.61875,-0.987917,-0.61875,-0.617083,-0.655417
